# Exploration de la qualité des données

Le but de ce notebook est de vérifier si les données collectées sont suffisamment fiables et pertinentes pour notre sujet.

En l'occurence, nous vérifierons dans un premier temps les valeurs manquantes, les doublons,... c'est-à-dire si les données sont colectées de manière correctes.

Ensuite, il s'agira de vérifier si ces données sont récentes, si elles concernent les joueurs et clubs qui nous intéressent et si les variables proposées sont suffisantes pour notre analyse.

Nous importons tout d'abord les packages nécessaires à notre première analyse exploratoire.

In [ ]:
import os
import pandas as pd
from pathlib import Path # pour trouver les chemins menant à nos bases de données

# Pour accéder à la dernière date de mise à jour
from kaggle.api.kaggle_api_extended import KaggleApi
from dotenv import load_dotenv


import json # pour analyser les fichiers json de Statsbomb

import requests
from datetime import datetime

c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
import os

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.append(os.path.abspath("../fonctions"))

# On importe toutes les fonctions dans le fichier imports.py
from utils import * # type: ignore

# 1. Transfermarkt

Suite à l'acquisition des données Transfermarkt, importées dans le dossier ../data/transfermarkt_datasets, nous pouvons alors commencer à analyser leur qualité.

## 1.1 Analyse des valeurs manquantes

In [3]:
chemin = "../data/transfermarkt_datasets"

nombre_NA_par_fichier(chemin) # type: ignore

c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\utils.py:24: DtypeWarning: Columns (0: number) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


                  fichier   lignes  colonnes       NA
0         appearances.csv  1862208        13        2
1               clubs.csv      796        17     1597
2          club_games.csv   173966        11   104264
3        competitions.csv       67        11       42
4           countries.csv      118         8        0
5               games.csv    86983        23    82284
6         game_events.csv  1242945        11  1782275
7        game_lineups.csv  3049833        10        3
8      national_teams.csv      118        17      127
9             players.csv    47702        26   188546
10  player_valuations.csv   616377         6    71820
11          transfers.csv   157186        10   115681


On regarde ensuite plus en détail dans chaque fichier les colonnes présentant le plus de données manquantes.

In [ ]:
# Pour tes fichiers Transfermarkt (CSV)
print("ANALYSE TRANSFERMARKT")
df_tm = analyze_missing_data("../data/transfermarkt_datasets", threshold=10) # type: ignore

--- ANALYSE TRANSFERMARKT ---


c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\utils.py:61: DtypeWarning: Columns (0: number) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)



Synthèse (> 10% de NA) :
  - club_games.csv : 2/11
  - clubs.csv : 2/17
  - competitions.csv : 3/11
  - game_events.csv : 2/11
  - games.csv : 5/23
  - national_teams.csv : 1/17
  - player_valuations.csv : 1/6
  - players.csv : 10/26
  - transfers.csv : 2/10

Fichier: club_games.csv [Format: CSV]
          colonne  lignes    NA  taux_NA_%
     own_position  173966 50444       29.0
opponent_position  173966 50444       29.0

Fichier: clubs.csv [Format: CSV]
           colonne  lignes  NA  taux_NA_%
total_market_value     796 796     100.00
        coach_name     796 706      88.69

Fichier: competitions.csv [Format: CSV]
             colonne  lignes  NA  taux_NA_%
         total_clubs      67  16      23.88
        country_name      67  13      19.40
domestic_league_code      67  13      19.40

Fichier: game_events.csv [Format: CSV]
         colonne  lignes      NA  taux_NA_%
player_assist_id 1242945 1059035       85.2
    player_in_id 1242945  630170       50.7

Fichier: games.csv [Fo

## 1.2 Analyse des doublons

Nous vérifions maintenant si les fichiers Transfermarkt présentent des doublons.

In [4]:
# Analyse avec le seuil de 60%
df_fuzzy = analyze_fuzzy_duplicates("../data/transfermarkt_datasets", similarity_threshold=1.0)

KeyboardInterrupt: 

In [ ]:
# On crée un dataframe vide pour le résumé des doublons
summary_duplicates = []

# On regarde par fichier les doublons et les taux de doublons
for file in folder.glob("*.csv"):
    df = pd.read_csv(file)
    
    # On regarde le nombre de doublons complets (lignes identiques)
    n_duplicates = df.duplicated().sum()
    
    # Les taux de doublons
    total_rows = len(df)
    dup_rate = (n_duplicates / total_rows) * 100 if total_rows > 0 else 0
    
    # Ce qu'on affiche
    summary_duplicates.append({
        "fichier": file.name,
        "lignes": total_rows,
        "doublons": n_duplicates,
        "taux_doublons_%": round(dup_rate, 2)
    })

# Les résultats des doublons
result_dup = pd.DataFrame(summary_duplicates)

# On trie par taux de doublons
result_dup = result_dup.sort_values(by="taux_doublons_%", ascending=False)

print("\nAnalyse des doublons par fichier :\n")
print(result_dup.to_string(index=False))

C:\Users\LouisHarle\AppData\Local\Temp\ipykernel_24308\4287159226.py:6: DtypeWarning: Columns (0: number) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)



Analyse des doublons par fichier :

              fichier  lignes  doublons  taux_doublons_%
      appearances.csv 1862208         0              0.0
            clubs.csv     796         0              0.0
       club_games.csv  173966         0              0.0
     competitions.csv      67         0              0.0
        countries.csv     118         0              0.0
            games.csv   86983         0              0.0
      game_events.csv 1242945         0              0.0
     game_lineups.csv 3049833         0              0.0
   national_teams.csv     118         0              0.0
          players.csv   47702         0              0.0
player_valuations.csv  616377         0              0.0
        transfers.csv  157186         0              0.0


## 1.3 La cohérence des données

Maintenant, nous devons vérifier si les données sont cohérentes. Nous regarderons dans un premier temps les types des variables dans chaque fichier .csv, puis les plages de valeurs pour les variables quantitatives.

In [ ]:
print("\nLes types de données par fichier\n")

# On affiche les types des colonnes
for file in folder.glob("*.csv"):
    df = pd.read_csv(file)
    
    print(f"\n{file.name}")
    print(df.dtypes)



Les types de données par fichier


appearances.csv
appearance_id               str
game_id                   int64
player_id                 int64
player_club_id            int64
player_current_club_id    int64
date                        str
player_name                 str
competition_id              str
yellow_cards              int64
red_cards                 int64
goals                     int64
assists                   int64
minutes_played            int64
dtype: object

clubs.csv
club_id                      int64
club_code                      str
name                           str
domestic_competition_id        str
total_market_value         float64
squad_size                   int64
average_age                float64
foreigners_number            int64
foreigners_percentage      float64
national_team_players        int64
stadium_name                   str
stadium_seats                int64
net_transfer_record            str
coach_name                     str
last_season      

C:\Users\LouisHarle\AppData\Local\Temp\ipykernel_24308\2758564645.py:4: DtypeWarning: Columns (0: number) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)



game_lineups.csv
game_lineups_id       str
date                  str
game_id             int64
player_id           int64
club_id             int64
player_name           str
type                  str
position              str
number             object
team_captain        int64
dtype: object

national_teams.csv
national_team_id           int64
name                         str
team_code                    str
country_id                 int64
country_name                 str
country_code                 str
confederation                str
team_image_url               str
squad_size                 int64
average_age              float64
foreigners_number          int64
foreigners_percentage    float64
total_market_value       float64
coach_name               float64
fifa_ranking               int64
last_season                int64
url                          str
dtype: object

players.csv
player_id                                 int64
first_name                                  str
last

In [ ]:
# Un résumé statistique sur les variables quantitatives

for file in folder.glob("*.csv"):
    df = pd.read_csv(file)
    
    print(f"\n{file.name}")
    
    num_cols = df.select_dtypes(include=["number"])
    
    if not num_cols.empty:
        print(num_cols.describe().transpose())


appearances.csv
                            count          mean            std        min  \
game_id                 1862208.0  3.286325e+06  751845.380006  2211607.0   
player_id               1862208.0  2.356550e+05  219774.730603       10.0   
player_club_id          1862208.0  3.315381e+03    9091.647935        1.0   
player_current_club_id  1862208.0  5.345555e+03   13576.894952       -1.0   
yellow_cards            1862208.0  1.457914e-01       0.363676        0.0   
red_cards               1862208.0  3.790661e-03       0.061452        0.0   
goals                   1862208.0  9.544208e-02       0.329904        0.0   
assists                 1862208.0  7.510654e-02       0.284768        0.0   
minutes_played          1862208.0  6.863815e+01      30.150312        1.0   

                              25%        50%        75%        max  
game_id                 2606535.0  3203656.0  3886492.0  4839901.0  
player_id                 61651.0   170767.0   342405.0  1510255.0  
playe

C:\Users\LouisHarle\AppData\Local\Temp\ipykernel_24308\301304749.py:2: DtypeWarning: Columns (0: number) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)



game_lineups.csv
                  count          mean            std        min        25%  \
game_id       3049833.0  3.462119e+06  756864.204133  2317258.0  2732178.0   
player_id     3049833.0  3.019559e+05  269605.278091       10.0    88685.0   
club_id       3049833.0  5.526004e+03   13085.594888        1.0      367.0   
team_captain  3049833.0  4.707176e-02       0.211792        0.0        0.0   

                    50%        75%        max  
game_id       3424029.0  4126828.0  4839901.0  
player_id      226151.0   428050.0  1527806.0  
club_id          1003.0     3426.0   134734.0  
team_captain        0.0        0.0        1.0  

national_teams.csv
                       count          mean           std       min  \
national_team_id       118.0  7.295695e+03  6.951144e+03    3262.0   
country_id             118.0  1.058814e+02  6.339824e+01       2.0   
squad_size             118.0  2.589831e+01  3.447754e+00      14.0   
average_age            118.0  2.692797e+01  1.31700

## 1.4 La couverture temporelle

Pour que nos données soient pertinentes, il faut qu'elles soient suffisamment récentes pour être analysées. Les prochains codes analysent donc la pertinence des données au niveau temporel.

In [ ]:
print("\nCouverture temporelle des datasets\n")

for file in folder.glob("*.csv"):
    df = pd.read_csv(file)
    
    # On cherche les colonnes présentant des dates
    date_cols = [col for col in df.columns if "date" in col.lower()]
    
    if not date_cols:
        continue
    
    print(f"\n{file.name}")
    
    # On cherche ici la date minimale et maximale puis on les affiche
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        
        min_date = df[col].min()
        max_date = df[col].max()
        
        print(f"  {col} : {min_date} → {max_date}")



Couverture temporelle des datasets


appearances.csv
  date : 2012-07-03 00:00:00 → 2026-03-22 00:00:00

games.csv
  date : 2006-06-09 00:00:00 → 2026-03-25 00:00:00

game_events.csv
  date : 2006-06-09 00:00:00 → 2026-03-25 00:00:00


C:\Users\LouisHarle\AppData\Local\Temp\ipykernel_24308\3590303887.py:4: DtypeWarning: Columns (0: number) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)



game_lineups.csv
  date : 2013-07-02 00:00:00 → 2026-03-23 00:00:00

players.csv
  date_of_birth : 1968-07-31 00:00:00 → 2011-02-23 00:00:00
  contract_expiration_date : 2000-05-31 00:00:00 → 2035-06-30 00:00:00

player_valuations.csv
  date : 2000-01-20 00:00:00 → 2026-03-30 00:00:00

transfers.csv
  transfer_date : 1993-07-01 00:00:00 → 2030-06-30 00:00:00


In [ ]:
# On cherche ici le nombre d'observations par année

for file in folder.glob("*.csv"):
    df = pd.read_csv(file)
    
    # Les colonnes présentant des dates
    date_cols = [col for col in df.columns if "date" in col.lower()]
    
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        
        print(f"\n{file.name} - {col}")
        # On compte le nombre d'observations par année
        print(df[col].dt.year.value_counts().sort_index())



appearances.csv - date
date
2012     71688
2013    129862
2014    125776
2015    132338
2016    131075
2017    135103
2018    126996
2019    128672
2020    114160
2021    147383
2022    129335
2023    150698
2024    145640
2025    149263
2026     44219
Name: count, dtype: int64

games.csv - date
date
2006       64
2008       31
2010       64
2012     3514
2013     5679
2014     5719
2015     5858
2016     5741
2017     5849
2018     5686
2019     5718
2020     4761
2021     6446
2022     5563
2023     6239
2024     7127
2025    10306
2026     2618
Name: count, dtype: int64

game_events.csv - date
date
2006       900
2008       393
2010       787
2012     44871
2013     71798
2014     72097
2015     75589
2016     73323
2017     74765
2018     73142
2019     74871
2020     70086
2021     98239
2022     87743
2023     99966
2024    116331
2025    166079
2026     41965
Name: count, dtype: int64


C:\Users\LouisHarle\AppData\Local\Temp\ipykernel_24308\3824261365.py:4: DtypeWarning: Columns (0: number) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)



game_lineups.csv - date
date
2013    125121
2014    205844
2015    212481
2016    207836
2017    216507
2018    203808
2019    208903
2020    185537
2021    252896
2022    217957
2023    252778
2024    289305
2025    384241
2026     86619
Name: count, dtype: int64

players.csv - date_of_birth
date_of_birth
1968.0       1
1970.0       2
1971.0       3
1972.0       5
1973.0       8
1974.0      20
1975.0      31
1976.0      50
1977.0      82
1978.0     130
1979.0     208
1980.0     280
1981.0     379
1982.0     404
1983.0     537
1984.0     622
1985.0     706
1986.0     886
1987.0     934
1988.0    1169
1989.0    1189
1990.0    1396
1991.0    1540
1992.0    1821
1993.0    2050
1994.0    2288
1995.0    2244
1996.0    2434
1997.0    2579
1998.0    2555
1999.0    2541
2000.0    2648
2001.0    2622
2002.0    2595
2003.0    2633
2004.0    2431
2005.0    2169
2006.0    1785
2007.0    1087
2008.0     464
2009.0     108
2010.0      16
2011.0       1
Name: count, dtype: int64

players.csv - contr

## 1.5 Biais de sélection

De potentiels biais peuvent apparaître dans les données, comme des joueurs ou des championnats surreprésentés. Nous pouvons ainsi vérifier si ce biais est présent dans nos données.

In [ ]:
# Nous analysons tout d'abord les joueurs

# Nous prenons les fichiers .csv qui nous intéressent ici : players.csv et appearances.csv
players = pd.read_csv(folder / "players.csv")
appearances = pd.read_csv(folder / "appearances.csv")

# La fréquence des joueurs dans les apparitions
player_counts = appearances["player_id"].value_counts().reset_index()
player_counts.columns = ["player_id", "count"]

# On merge avec les noms des joueurs
player_counts = player_counts.merge(
    players[["player_id", "name"]],
    on="player_id",
    how="left"
)

print("\nTop 20 des joueurs les plus présents :\n")
print(player_counts[["name", "count"]].head(20))


Top 20 des joueurs les plus présents :

                  name  count
0   Robert Lewandowski    656
1    Antoine Griezmann    622
2                 Koke    616
3          Dani Parejo    608
4   Henrikh Mkhitaryan    584
5           Edin Dzeko    582
6                Pedro    581
7     Thibaut Courtois    581
8          Luka Modrić    580
9         Ivan Rakitic    578
10     Virgil van Dijk    576
11         Dušan Tadić    569
12      Marten de Roon    568
13       Mohamed Salah    568
14           Jan Oblak    565
15     Bruno Fernandes    562
16      Bernardo Silva    562
17       Dries Mertens    560
18        David de Gea    559
19       Álvaro Morata    558


In [ ]:
print("\nRésumé distribution apparitions :")
print(player_counts["count"].describe())


Résumé distribution apparitions :
count    28665.000000
mean        64.964521
std         86.593096
min          1.000000
25%          6.000000
50%         28.000000
75%         89.000000
max        656.000000
Name: count, dtype: float64


In [ ]:
# Nous analysons ensuite les championnats

# Le fichier games.csv est alors celui qui nous intéresse
games = pd.read_csv(folder / "games.csv")

# On compte le nombre de matches par championnat
league_counts = games["competition_id"].value_counts()

print("\nTop compétitions :\n")
print(league_counts.head(20))



Top compétitions :

competition_id
GB1     5249
IT1     5240
ES1     5230
FR1     4933
TR1     4553
L1      4221
NL1     4156
PO1     4087
BE1     3550
RU1     3296
GR1     3085
SC1     2891
ELQ     2677
UKR1    2626
EL      2616
DK1     2312
FAC     2008
CL      1859
CDR     1676
RUP     1641
Name: count, dtype: int64


In [ ]:
# Pour y voir plus clair (les noms des championnats), nous mergeons ensuite avec competitions.csv

competitions = pd.read_csv(folder / "competitions.csv")

merged = games.merge(competitions, left_on="competition_id", right_on="competition_id")

print("\nRépartition par championnat :\n")
print(merged["name"].value_counts())



Répartition par championnat :

name
premier-liga                         5922
premier-league                       5249
serie-a                              5240
laliga                               5230
ligue-1                              4933
super-lig                            4553
bundesliga                           4529
eredivisie                           4156
liga-portugal                        4087
jupiler-pro-league                   3550
super-league-1                       3085
scottish-premiership                 2891
superliga                            2752
uefa-europa-league-qualifying        2677
uefa-europa-league                   2616
fa-cup                               2008
uefa-champions-league                1859
copa-del-rey                         1676
russian-cup                          1641
scottish-fa-cup                      1362
uefa-conference-league-qualifiers    1344
knvb-beker                           1265
uefa-champions-league-qualifying     12

## 1.6 La date de dernière mise à jour

Pour constater si nos données sont mises à jour régulièrement, nous pouvons regarder la date de dernière mise à jour.

In [8]:
# On charge les variables du fichier .env (identifiant et clé API Kaggle)
load_dotenv(dotenv_path="../.env")

os.environ['KAGGLE_USERNAME'] = os.getenv('KAGGLE_USERNAME')
os.environ['KAGGLE_API_TOKEN'] = os.getenv('KAGGLE_API_TOKEN')

# On charge l'API de Kaggle
api = KaggleApi()

# On recherche le dataset
dataset_query = 'davidcariboo/player-scores'

# On récupère le dataset issu du Kaggle
datasets = api.dataset_list(search=dataset_query)

# On trouve la date de dernière mise à jour
for ds in datasets:
    if ds.ref == dataset_query:
        # Récupération de la date (gestion des différentes versions du package)
        date_maj = getattr(ds, 'lastUpdated', None) or getattr(ds, 'last_updated', "Date inconnue")
        
        print(f"Dataset trouvé : {ds.ref}")
        print(f"Dernière mise à jour : {date_maj}")
        break


Dataset trouvé : davidcariboo/player-scores
Dernière mise à jour : 2026-04-01 09:41:00.553000


# 2. FBref

## 2.1 Analyse des valeurs manquantes

Suite à l'acquisition des données FBref, importées dans le dossier ../data/fbref_datasets, nous pouvons alors commencer à analyser leur qualité.

In [5]:
chemin = "../data/fbref_datasets"

nombre_NA_par_fichier(chemin) # type: ignore

                            fichier  lignes  colonnes     NA
0        players_data-2025_2026.csv    2751       102  64346
1  players_data_light-2025_2026.csv    2751        53  37792


On regarde ensuite plus en détail dans chaque fichier les colonnes présentant le plus de données manquantes.

In [ ]:
# On créer un summary vide et un dictionnaire total_cols vide
summary = []
total_cols = {}

# On analyse tous les fichiers .csv issus de FBref
for file in folder.glob("*.csv"):
    df = pd.read_csv(file)
    
    # On stocke le nombre total de colonnes
    total_cols[file.name] = len(df.columns)
    
    # Pour chaque colonne, on compte le nombre de lignes et de NA, et par conséquent le taux de NA
    for col in df.columns:
        total = len(df)
        na_count = df[col].isna().sum()
        na_rate = (na_count / total) * 100 if total > 0 else 0
        
        # On filtre les colonnes dont le taux de NA est supérieur à 10%
        if na_rate > 10:
            summary.append({
                "fichier": file.name,
                "colonne": col,
                "lignes": total,
                "NA": na_count,
                "taux_NA_%": round(na_rate, 2)
            })

# On stocke les résultats dans un dataframe
result = pd.DataFrame(summary)

# On compte le nombre de colonnes problématiques par fichier
count_per_file = result.groupby("fichier")["colonne"].count()

print("\nColonnes avec >10% de NA (problématiques / total) :\n")

# On affiche le nombre de colonnes problématiques par fichier
for fichier, count in count_per_file.items():
    total = total_cols[fichier]
    print(f"{fichier} : {count}/{total}")


# On trie les colonnes problématiques dans l'ordre décroissant des taux de NA
result_sorted = result.sort_values(by=["fichier", "taux_NA_%"], ascending=[True, False])


print("\nDétail des colonnes problématiques (trié par fichier) :\n")

# On affiche les colonnes problématiques par fichier
for fichier, df_file in result_sorted.groupby("fichier"):
    print(f"\n{fichier}")
    print(df_file[["colonne", "lignes", "NA", "taux_NA_%"]].to_string(index=False))


Colonnes avec >10% de NA (problématiques / total) :

players_data-2025_2026.csv : 29/102
players_data_light-2025_2026.csv : 17/53

Détail des colonnes problématiques (trié par fichier) :


players_data-2025_2026.csv
            colonne  lignes   NA  taux_NA_%
                CS%    2731 2563      93.85
              Save%    2731 2561      93.78
    Rk_stats_keeper    2731 2560      93.74
Nation_stats_keeper    2731 2560      93.74
   Pos_stats_keeper    2731 2560      93.74
  Comp_stats_keeper    2731 2560      93.74
   Age_stats_keeper    2731 2560      93.74
  Born_stats_keeper    2731 2560      93.74
    MP_stats_keeper    2731 2560      93.74
Starts_stats_keeper    2731 2560      93.74
   Min_stats_keeper    2731 2560      93.74
   90s_stats_keeper    2731 2560      93.74
                 GA    2731 2560      93.74
               GA90    2731 2560      93.74
               SoTA    2731 2560      93.74
              Saves    2731 2560      93.74
                  W    2731 2560   

## 2.2 Analyse des doublons

Nous vérifions maintenant si les fichiers FBref présentent des doublons.

In [ ]:
# On crée un dataframe vide pour le résumé des doublons
summary_duplicates = []

# On regarde par fichier les doublons et les taux de doublons
for file in folder.glob("*.csv"):
    df = pd.read_csv(file)
    
    # On regarde le nombre de doublons complets (lignes identiques)
    n_duplicates = df.duplicated().sum()
    
    # Les taux de doublons
    total_rows = len(df)
    dup_rate = (n_duplicates / total_rows) * 100 if total_rows > 0 else 0
    
    # Ce qu'on affiche
    summary_duplicates.append({
        "fichier": file.name,
        "lignes": total_rows,
        "doublons": n_duplicates,
        "taux_doublons_%": round(dup_rate, 2)
    })

# Les résultats des doublons
result_dup = pd.DataFrame(summary_duplicates)

# On trie par taux de doublons
result_dup = result_dup.sort_values(by="taux_doublons_%", ascending=False)

print("\nAnalyse des doublons par fichier :\n")
print(result_dup.to_string(index=False))


Analyse des doublons par fichier :

                         fichier  lignes  doublons  taux_doublons_%
      players_data-2025_2026.csv    2731         0              0.0
players_data_light-2025_2026.csv    2731         0              0.0


## 2.3 La cohérence des données

Maintenant, nous devons vérifier si les données sont cohérentes. Nous regarderons dans un premier temps les types des variables dans chaque fichier .csv, puis les plages de valeurs pour les variables quantitatives.

In [ ]:
print("\nLes types de données par fichier\n")

# On affiche les types des colonnes
for file in folder.glob("*.csv"):
    df = pd.read_csv(file)
    
    print(f"\n{file.name}")
    print(df.dtypes)


Les types de données par fichier


players_data-2025_2026.csv
Rk                             int64
Player                           str
Nation                           str
Pos                              str
Squad                            str
Comp                             str
Age                          float64
Born                         float64
MP                             int64
Starts                         int64
Min                            int64
90s                          float64
Gls                            int64
Ast                            int64
G+A                            int64
G-PK                           int64
PK                             int64
PKatt                          int64
CrdY                           int64
CrdR                           int64
G+A-PK                       float64
Rk_stats_keeper              float64
Nation_stats_keeper              str
Pos_stats_keeper                 str
Comp_stats_keeper                str
Age_stats_ke

In [ ]:
# Un résumé statistique des variables quantitatives

for file in folder.glob("*.csv"):
    df = pd.read_csv(file)
    
    print(f"\n{file.name}")
    
    num_cols = df.select_dtypes(include=["number"])
    
    if not num_cols.empty:
        print(num_cols.describe().transpose())


players_data-2025_2026.csv
                            count         mean         std      min       25%  \
Rk                         2731.0  1366.000000  788.516117     1.00   683.500   
Age                        2730.0    25.675458    4.560993    16.00    22.000   
Born                       2730.0  1999.664103    4.562300  1983.00  1997.000   
MP                         2731.0    16.135848    9.316270     1.00     8.000   
Starts                     2731.0    11.455145    9.295421     0.00     3.000   
Min                        2731.0  1028.034420  794.819728     1.00   292.000   
90s                        2731.0    11.422776    8.832459     0.00     3.200   
Gls                        2731.0     1.393263    2.471051     0.00     0.000   
Ast                        2731.0     0.960454    1.543951     0.00     0.000   
G+A                        2731.0     2.353717    3.460521     0.00     0.000   
G-PK                       2731.0     1.259612    2.155063     0.00     0.000   


## 2.4 La couverture temporelle

Pour que nos données soient pertinentes, il faut qu'elles soient suffisamment récentes pour être analysées. Nous savons que les données analysées dans ces fichiers .csv proviennent de la saison 2025-2026.

## 2.5 Biais de sélection

De potentiels biais peuvent apparaître dans les données, comme des joueurs ou des championnats surreprésentés. Nous pouvons ainsi vérifier si ce biais est présent dans nos données.

In [ ]:
# On charge le fichier
players_data = pd.read_csv(folder / "players_data-2025_2026.csv")

# La fréquence des joueurs basée sur le nombre de matches joués
player_counts = df.groupby("Player")["MP"].sum().reset_index()

# On renomme MP que je ne trouvais pas très parlant
player_counts = player_counts.rename(columns={"MP": "Matches joués"})

# On trie les joueurs dans l'ordre décroissant des matches joués
player_counts = player_counts.sort_values("Matches joués", ascending=False)

print("\nTop 20 des joueurs les plus présents :\n")
print(player_counts.head(20))



Top 20 des joueurs les plus présents :

                  Player  Matches joués
2446             Vitinha             55
1844    Nicolás González             45
625        Donyell Malen             32
1233          João Pedro             31
1118    Jesper Karlström             31
2384           Trai Hume             31
1092          Jean Butez             31
328           Bernd Leno             31
331                 Beto             31
2335       Tiago Gabriel             31
1864   Nikola Milenković             31
1077        Jarrod Bowen             31
1430     Lorenzo Colombo             31
2432      Victor Nelsson             31
1503           Luke Shaw             31
558           David Raya             31
560         David de Gea             31
652         Elia Caprile             31
300      Bart Verbruggen             31
1766  Morgan Gibbs-White             31


In [ ]:
print("\nRésumé distribution apparitions :")
print(player_counts["Matches joués"].describe())


Résumé distribution apparitions :
count    2579.000000
mean       17.086855
std         9.141262
min         1.000000
25%         9.000000
50%        19.000000
75%        25.000000
max        55.000000
Name: Matches joués, dtype: float64


In [ ]:
# Nous analysons ensuite les championnats

# On compte le nombre de matches par championnat
league_counts = players_data["Comp"].value_counts().reset_index()

# On renomme la colonne
league_counts.columns = ["Comp", "joueurs"]

# On trie dans l'ordre décroissant
league_counts = league_counts.sort_values("joueurs", ascending=False)

print("\nTop compétitions :\n")
print(league_counts.head(20))



Top compétitions :

                 Comp  joueurs
0          it Serie A      582
1          es La Liga      581
2          fr Ligue 1      540
3  eng Premier League      535
4       de Bundesliga      493


## 2.6 La date de dernière mise à jour

Pour constater si nos données sont mises à jour régulièrement, nous pouvons regarder la date de dernière mise à jour.

In [ ]:
# On charge les variables du fichier .env (identifiant et clé API Kaggle)
load_dotenv(dotenv_path="../.env")

os.environ['KAGGLE_USERNAME'] = os.getenv('KAGGLE_USERNAME')
os.environ['KAGGLE_API_TOKEN'] = os.getenv('KAGGLE_API_TOKEN')

# On charge l'API de Kaggle
api = KaggleApi()

# On recherche le dataset
dataset_query = 'hubertsidorowicz/football-players-stats-2025-2026'

# On récupère le dataset issu du Kaggle
datasets = api.dataset_list(search=dataset_query)

# On trouve la date de dernière mise à jour
for ds in datasets:
    if ds.ref == dataset_query:
        # Récupération de la date (gestion des différentes versions du package)
        date_maj = getattr(ds, 'lastUpdated', None) or getattr(ds, 'last_updated', "Date inconnue")
        
        print(f"Dataset trouvé : {ds.ref}")
        print(f"Dernière mise à jour : {date_maj}")
        break


Dataset trouvé : hubertsidorowicz/football-players-stats-2025-2026
Dernière mise à jour : 2026-04-21 12:40:14.337000


# 3. Statsbomb

## 3.1 Regroupement des données dans un fichier .csv

Suite à l'acquisition des données Statsbomb, importées dans le dossier ../data/statsbomb_datasets, nous pouvons alors commencer à analyser leur qualité.

Dans un premier temps, nous transformons les données .json en données centralisées dans un fichier .csv, que ce soit pour les lineups ou les events.

In [ ]:
# Pour le dossier lineups
folder = Path("../data/statsbomb_datasets/data/lineups")

# On prend tous les fichiers .json et on crée une liste vide
json_files = list(folder.glob("*.json"))
all_lineups = [] 


# Boucle de lecture sur les fichiers
for file_path in json_files:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Extraction et normalisation
    df = pd.json_normalize(
        data, 
        record_path=['lineup'], 
        meta=['team_name', 'team_id'],
        errors='ignore'
    )
    
    # Ajout de l'ID du match
    df['match_id'] = file_path.stem 
    all_lineups.append(df)

# On crée un fichier CSV unique
final_df = pd.concat(all_lineups, ignore_index=True)

# Sauvegarde finale
output_path = folder / "all_lineups_combined.csv"
final_df.to_csv(output_path, index=False, encoding='utf-8')

print(f"Fichier créé : {output_path.name}")
print(f"Nombre total de joueurs répertoriés : {len(final_df)}")

Fichier créé : all_lineups_combined.csv
Nombre total de joueurs répertoriés : 131901


In [41]:
# Pour le dossier events
folder = Path("../data/statsbomb_datasets/data/events")

# On prend tous les fichiers .json
json_files = list(folder.glob("*.json"))
output_path = folder / "all_events_combined.feather"

# Boucle sur l'ensemble des fichiers .json
for i, file_path in enumerate(json_files):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # On aplatit le JSON
    df = pd.json_normalize(data)
    
    # On ajoute l'ID du match
    df['match_id'] = file_path.stem 
    
    # Écriture sur le disque
    if i == 0:
        df.to_feather(output_path, index=False, mode='w', encoding='utf-8')
    else:
        df.to_feather(output_path, index=False, mode='a', header=False, encoding='utf-8')
    
    # Suivi de progression toutes les 100 étapes
    if (i + 1) % 100 == 0:
        print(f"{i + 1}/{len(json_files)} fichiers compilés...")

print(f"Fichier global créé : {output_path.name}")

ImportError: `Import pyarrow` failed.  Use pip or conda to install the pyarrow package.

In [ ]:
# Pour le fichier competitions.json
folder = Path("../data/statsbomb_datasets/data")
json_files = list(folder.glob("*.json"))

# On définit le nouvel emplacement de sortie
output_folder = Path("../data/statsbomb_datasets/data")

# Le chemin complet du fichier final
output_path = output_folder / "competitions_statsbomb.csv"

# Liste pour stocker les DataFrames
all_dfs = []

for file_path in json_files:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        all_dfs.append(pd.DataFrame(data))

final_df = pd.concat(all_dfs, ignore_index=True)

final_df.to_csv(output_path, index=False, encoding='utf-8')
print(f"Le fichier a été enregistré ici : {output_path.resolve()}")

Le fichier a été enregistré ici : C:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\predict_vm_football\data\statsbomb_datasets\data\competitions_statsbomb.csv


## 3.2 Analyse des valeurs manquantes

Nous pouvons ensuite analyser les valeurs manquantes de ces deux fichiers .csv.

In [ ]:
# On cherche le dossier dans lequel sont les bases de données
folder = Path("../data/statsbomb_datasets/data")

# On prend seulement ce fichier-ci pour le moment
file = folder / "all_lineups_combined.csv"
lineups = pd.read_csv(file, low_memory=False)

# On crée un dataframe vide pour ensuite produire un résumé des données
summary = []

# On regarde le nombre de lignes, colonnes et NA
summary.append({
    "fichier": file.name,
    "lignes": len(lineups),
    "colonnes": len(lineups.columns),
    "NA": lineups.isna().sum().sum()
})

# Affichage
print(pd.DataFrame(summary))


                    fichier  lignes  colonnes     NA
0  all_lineups_combined.csv  131901        11  81729


On regarde ensuite plus en détail dans chaque fichier les colonnes présentant le plus de données manquantes.

In [ ]:
file_path = folder / file

# On créer un summary vide et un dictionnaire total_cols vide
summary = []
total_cols = {}

total_rows = len(lineups)
total_cols = len(lineups.columns)


for col in lineups.columns:
    na_count = lineups[col].isna().sum()
    na_rate = (na_count / total_rows) * 100 if total_rows > 0 else 0
    
    # On filtre les colonnes dont le taux de NA est supérieur à 10%
    if na_rate > 10:
        summary.append({
            "fichier": file,
            "colonne": col,
            "lignes": total_rows,
            "NA": na_count,
            "taux_NA_%": round(na_rate, 2)
        })


# On stocke les résultats dans un dataframe
result = pd.DataFrame(summary)

print(f"Colonnes avec >10% de NA : {len(result)}/{total_cols}")
print(result[["colonne", "lignes", "NA", "taux_NA_%"]])

Colonnes avec >10% de NA : 1/11
           colonne  lignes     NA  taux_NA_%
0  player_nickname  131901  81703      61.94


## 3.3 Analyse des doublons

Nous vérifions maintenant si les fichiers Statsbomb présentent des doublons.

In [ ]:
# Calcul des doublons
n_duplicates = lineups.duplicated().sum()
total_rows = len(lineups)
dup_rate = (n_duplicates / total_rows) * 100 if total_rows > 0 else 0

# Création du résumé
summary_duplicates = [{
    "fichier": file_path.name,
    "lignes": total_rows,
    "doublons": n_duplicates,
    "taux_doublons_%": round(dup_rate, 2)
}]


doublons = pd.DataFrame(summary_duplicates)
print("\nAnalyse des doublons pour all_lineups_combined.csv :\n")
print(doublons.to_string(index=False))


Analyse des doublons pour all_lineups_combined.csv :

                 fichier  lignes  doublons  taux_doublons_%
all_lineups_combined.csv  131901         0              0.0


## 3.4 La cohérence des données

Maintenant, nous devons vérifier si les données sont cohérentes. Nous regarderons dans un premier temps les types des variables dans chaque fichier .csv, puis les plages de valeurs pour les variables quantitatives.

In [ ]:
print("\nLes types de données pour all_lineups_combined.csv\n")

print(lineups.dtypes)


Les types de données pour all_lineups_combined.csv

player_id            int64
player_name            str
player_nickname        str
jersey_number        int64
cards                  str
positions              str
country.id         float64
country.name           str
team_name              str
team_id              int64
match_id             int64
dtype: object


In [ ]:
# Sélection des colonnes numériques
num_cols = lineups.select_dtypes(include=["number"])

if not num_cols.empty:
    # On affiche la transposée pour une meilleure lecture (colonnes en lignes)
    print(num_cols.describe().transpose())
else:
    print("Aucune colonne numérique trouvée dans ce fichier.")

                  count          mean           std     min        25%  \
player_id      131901.0  3.544764e+04  7.713198e+04  2935.0     5615.0   
jersey_number  131901.0  1.614910e+01  1.398939e+01     0.0        7.0   
country.id     131888.0  1.214309e+02  7.238412e+01     3.0       68.0   
team_id        131901.0  8.202458e+02  1.753747e+03     1.0      174.0   
match_id       131901.0  3.065607e+06  1.501069e+06  7298.0  3754036.0   

                     50%        75%        max  
player_id         9731.0    26076.0   482216.0  
jersey_number       14.0       22.0     1000.0  
country.id         105.0      203.0      255.0  
team_id            224.0      859.0    29167.0  
match_id       3825768.0  3889149.0  4020846.0  


## 3.5 La couverture temporelle

Pour que nos données soient pertinentes, il faut qu'elles soient suffisamment récentes pour être analysées. Les prochains codes analysent donc la pertinence des données au niveau temporel.

In [ ]:
# On définit le dossier où se trouve ton nouveau CSV
folder = Path("../data/statsbomb_datasets/data")
file_path = folder / "competitions_statsbomb.csv"

print("\nCouverture temporelle du dataset des compétitions\n")

competitions = pd.read_csv(file_path)
    
# On élargit la recherche : colonnes avec "date", "updated" ou "available"
# Car StatsBomb utilise 'match_updated' pour l'horodatage
date_cols = [col for col in df.columns if any(x in col.lower() for x in ["date", "updated", "available"])]

if date_cols:
    print(f"Fichier : {file_path.name}")
        
    for col in date_cols:
        # Conversion en datetime
        competitions[col] = pd.to_datetime(df[col], errors="coerce")
            
        min_date = competitions[col].min()
        max_date = competitions[col].max()
            
        # Affichage propre (on retire les nanosecondes si elles existent)
        print(f"  {col} : {min_date} → {max_date}")
            
else:
    print(f"Le fichier {file_path} est introuvable.")


Couverture temporelle du dataset des compétitions

Fichier : competitions_statsbomb.csv
  match_updated : 2023-06-18 01:55:53.343752 → 2025-07-28 14:19:20.467348
  match_updated_360 : 2021-06-12 16:17:31.694000 → 2025-07-29 16:03:07.355174
  match_available_360 : 2024-02-13 13:30:52.820588 → 2025-07-29 16:03:07.355174
  match_available : 2023-06-18 01:55:53.343752 → 2025-07-28 14:19:20.467348


## 3.6 Biais de sélection

De potentiels biais peuvent apparaître dans les données, comme des joueurs ou des championnats surreprésentés. Nous pouvons ainsi vérifier si ce biais est présent dans nos données.

In [ ]:
# Nous analysons tout d'abord les joueurs

# La fréquence des joueurs dans les apparitions
player_counts = lineups.groupby(["player_id", "player_name"]).size().reset_index(name="composition")

# Tri par nombre d'apparitions (ordre décroissant)
player_counts = player_counts.sort_values(by="composition", ascending=False)


print("\nTop 20 des joueurs les plus présents (dans les compositions) :\n")
print(player_counts[["player_name", "composition"]].head(20).to_string(index=False))


Top 20 des joueurs les plus présents (dans les compositions) :

                    player_name  composition
 Lionel Andrés Messi Cuccittini          604
       Sergio Busquets i Burgos          420
           Andrés Iniesta Luján          361
          Gerard Piqué Bernabéu          358
               Jordi Alba Ramos          284
         Xavier Hernández Creus          267
          Víctor Valdés Arribas          257
          Daniel Alves da Silva          250
          Marc-André ter Stegen          237
    Javier Alejandro Mascherano          221
                   Ivan Rakitić          217
Pedro Eliezer Rodríguez Ledesma          216
       Luis Alberto Suárez Díaz          204
         Sergi Roberto Carnicer          198
       Carles Puyol i Saforcada          187
             Samuel Yves Umtiti          162
  Neymar da Silva Santos Junior          158
          Adriano Correia Claro          152
              Antoine Griezmann          145
                   Seydou Kéita    

In [ ]:
print("\nRésumé distribution apparitions :")
print(player_counts["composition"].describe())


Résumé distribution apparitions :
count    10814.000000
mean        12.197244
std         18.564377
min          1.000000
25%          2.000000
50%          4.000000
75%         17.000000
max        604.000000
Name: composition, dtype: float64


In [ ]:
# Nous analysons ensuite les équipes

# On utilise donc 'team_name' pour voir les équipes les plus actives

# Le nombre de matchs uniques par équipe :
league_analysis = lineups.groupby("team_name")["match_id"].nunique().reset_index(name="nb_matchs")

# Tri par nombre de matchs
league_analysis = league_analysis.sort_values(by="nb_matchs", ascending=False)

print("\nTop 20 des équipes ayant le plus de matchs répertoriés :\n")
print(league_analysis.head(20).to_string(index=False))



Top 20 des équipes ayant le plus de matchs répertoriés :

                 team_name  nb_matchs
                 Barcelona        532
       Paris Saint-Germain         95
         Manchester United         79
                   Arsenal         76
               Real Madrid         71
          Bayer Leverkusen         68
           Atlético Madrid         68
                   Sevilla         66
                  Valencia         65
             Athletic Club         64
                  Espanyol         63
               Aston Villa         62
                Villarreal         61
                    Getafe         61
       Manchester City WFC         59
             Real Sociedad         59
                Real Betis         58
Brighton & Hove Albion WFC         57
               Chelsea FCW         57
               Arsenal WFC         57


In [ ]:
# Nous analysons ensuite les championnats à partir du fichier généré

# On compte le nombre de saisons disponibles par championnat
# On utilise "competition_name" pour que l'affichage soit explicite
league_counts = competitions["competition_name"].value_counts()

print("\nNombre de saisons disponibles par compétition :\n")
print(league_counts.head(20))


Nombre de saisons disponibles par compétition :

competition_name
Champions League           18
La Liga                    18
FIFA World Cup              8
Copa del Rey                3
FA Women's Super League     3
Ligue 1                     3
1. Bundesliga               2
Liga Profesional            2
Premier League              2
Serie A                     2
UEFA Euro                   2
UEFA Women's Euro           2
Women's World Cup           2
African Cup of Nations      1
Copa America                1
FIFA U20 World Cup          1
Indian Super league         1
Major League Soccer         1
North American League       1
NWSL                        1
Name: count, dtype: int64


## 3.7 La date de dernière mise à jour

Pour constater si nos données sont mises à jour régulièrement, nous pouvons regarder la date de dernière mise à jour.

In [ ]:
# On convertit la variable d'update en format date
competitions['match_updated'] = pd.to_datetime(df['match_updated'])

# On trouve la date la plus récente
derniere_maj = competitions['match_updated'].max()

print(f"La dernière mise à jour enregistrée dans ce fichier date du : {derniere_maj}")

La dernière mise à jour enregistrée dans ce fichier date du : 2025-07-28 14:19:20.467348


# 4. football-data.co.uk

## 4.1 Analyse des valeurs manquantes

Suite à l'acquisition des données football-data, importées dans le dossier ../data sous le nom football_data_2526.csv, nous pouvons alors commencer à analyser leur qualité.

In [6]:
chemin = "../data"

nombre_NA_par_fichier(chemin) # type: ignore

             fichier  lignes  colonnes      NA
0  football_data.csv    5010       163  247125


c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\utils.py:24: DtypeWarning: Columns (0: Referee) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


On regarde ensuite plus en détail dans chaque fichier les colonnes présentant le plus de données manquantes.

In [ ]:
# On créer un summary vide et un dictionnaire total_cols vide
summary = []
total_cols = {}

# On analyse tous les fichiers .csv issus de football-data.co.uk
for file in folder.glob("*.csv"):
    df = pd.read_csv(file)
    
    # On stocke le nombre total de colonnes
    total_cols[file.name] = len(df.columns)
    
    # Pour chaque colonne, on compte le nombre de lignes et de NA, et par conséquent le taux de NA
    for col in df.columns:
        total = len(df)
        na_count = df[col].isna().sum()
        na_rate = (na_count / total) * 100 if total > 0 else 0
        
        # On filtre les colonnes dont le taux de NA est supérieur à 10%
        if na_rate > 10:
            summary.append({
                "fichier": file.name,
                "colonne": col,
                "lignes": total,
                "NA": na_count,
                "taux_NA_%": round(na_rate, 2)
            })

# On stocke les résultats dans un dataframe
result = pd.DataFrame(summary)

# On compte le nombre de colonnes problématiques par fichier
count_per_file = result.groupby("fichier")["colonne"].count()

print("\nColonnes avec >10% de NA (problématiques / total) :\n")

# On affiche le nombre de colonnes problématiques par fichier
for fichier, count in count_per_file.items():
    total = total_cols[fichier]
    print(f"{fichier} : {count}/{total}")


# On trie les colonnes problématiques dans l'ordre décroissant des taux de NA
result_sorted = result.sort_values(by=["fichier", "taux_NA_%"], ascending=[True, False])


print("\nDétail des colonnes problématiques (trié par fichier) :\n")

# On affiche les colonnes problématiques par fichier
for fichier, df_file in result_sorted.groupby("fichier"):
    print(f"\n{fichier}")
    print(df_file[["colonne", "lignes", "NA", "taux_NA_%"]].to_string(index=False))


Colonnes avec >10% de NA (problématiques / total) :

football_data_2526.csv : 27/133

Détail des colonnes problématiques (trié par fichier) :


football_data_2526.csv
colonne  lignes   NA  taux_NA_%
Referee    1468 1149      78.27
 PC>2.5    1468  577      39.31
 PC<2.5    1468  577      39.31
  P>2.5    1468  576      39.24
  P<2.5    1468  576      39.24
   PSCH    1468  570      38.83
   PSCD    1468  570      38.83
   PSCA    1468  570      38.83
  PCAHH    1468  569      38.76
  PCAHA    1468  569      38.76
    PSH    1468  566      38.56
    PSD    1468  566      38.56
    PSA    1468  566      38.56
   PAHH    1468  566      38.56
   PAHA    1468  566      38.56
   CLCH    1468  407      27.72
   CLCD    1468  407      27.72
   CLCA    1468  407      27.72
   LBCH    1468  407      27.72
   LBCD    1468  407      27.72
   LBCA    1468  407      27.72
    CLH    1468  394      26.84
    CLD    1468  394      26.84
    CLA    1468  394      26.84
    LBH    1468  394      26.84


## 4.2 Analyse des doublons

Nous vérifions maintenant si le fichier présente des doublons.

In [ ]:
# On crée un dataframe vide pour le résumé des doublons
summary_duplicates = []

# On regarde par fichier les doublons et les taux de doublons
for file in folder.glob("*.csv"):
    df = pd.read_csv(file)
    
    # On regarde le nombre de doublons complets (lignes identiques)
    n_duplicates = df.duplicated().sum()
    
    # Les taux de doublons
    total_rows = len(df)
    dup_rate = (n_duplicates / total_rows) * 100 if total_rows > 0 else 0
    
    # Ce qu'on affiche
    summary_duplicates.append({
        "fichier": file.name,
        "lignes": total_rows,
        "doublons": n_duplicates,
        "taux_doublons_%": round(dup_rate, 2)
    })

# Les résultats des doublons
result_dup = pd.DataFrame(summary_duplicates)

# On trie par taux de doublons
result_dup = result_dup.sort_values(by="taux_doublons_%", ascending=False)

print("\nAnalyse des doublons par fichier :\n")
print(result_dup.to_string(index=False))


Analyse des doublons par fichier :

               fichier  lignes  doublons  taux_doublons_%
football_data_2526.csv    1468         0              0.0


## 4.3 la cohérence des données

Maintenant, nous devons vérifier si les données sont cohérentes. Nous regarderons dans un premier temps les types des variables dans chaque fichier .csv, puis les plages de valeurs pour les variables quantitatives.

In [ ]:
print("\nLes types de données par fichier\n")

# On affiche les types pour chaque colonne
for file in folder.glob("*.csv"):
    df = pd.read_csv(file)
    
    print(f"\n{file.name}")
    print(df.dtypes)



Les types de données par fichier


football_data_2526.csv
Div             str
Date            str
Time            str
HomeTeam        str
AwayTeam        str
             ...   
AvgCAHA     float64
BFECAHH     float64
BFECAHA     float64
league          str
Referee         str
Length: 133, dtype: object


In [ ]:
# Un résumé statistique pour les variables quantitatives

for file in folder.glob("*.csv"):
    df = pd.read_csv(file)
    
    print(f"\n{file.name}")
    
    num_cols = df.select_dtypes(include=["number"])
    
    if not num_cols.empty:
        print(num_cols.describe().transpose())


football_data_2526.csv
          count       mean       std   min    25%    50%    75%    max
FTHG     1468.0   1.532698  1.270351  0.00   1.00   1.00   2.00   8.00
FTAG     1468.0   1.220027  1.140424  0.00   0.00   1.00   2.00   7.00
HTHG     1468.0   0.662125  0.806591  0.00   0.00   0.00   1.00   5.00
HTAG     1468.0   0.525204  0.753120  0.00   0.00   0.00   1.00   5.00
HS       1468.0  13.765668  5.255139  1.00  10.00  13.00  17.00  35.00
...         ...        ...       ...   ...    ...    ...    ...    ...
MaxCAHA  1468.0   1.954748  0.093320  1.70   1.88   1.95   2.03   2.35
AvgCAHH  1468.0   1.879469  0.084849  1.69   1.81   1.88   1.94   2.15
AvgCAHA  1468.0   1.884605  0.085744  1.68   1.82   1.88   1.95   2.10
BFECAHH  1468.0   1.992003  0.098259  1.72   1.91   1.99   2.06   2.32
BFECAHA  1468.0   1.995518  0.100799  1.74   1.92   1.99   2.07   2.80

[124 rows x 8 columns]


## 4.4 La couverture temporelle

Pour que nos données soient pertinentes, il faut qu'elles soient suffisamment récentes pour être analysées. Les prochains codes analysent donc la pertinence des données au niveau temporel.

In [ ]:
print("\nCouverture temporelle des datasets\n")

for file in folder.glob("*.csv"):
    df = pd.read_csv(file)
    
    # On cherche les colonnes présentant des dates
    date_cols = [col for col in df.columns if "date" in col.lower()]
    
    if not date_cols:
        continue
    
    print(f"\n{file.name}")
    
    # On cherche ici la date minimale et maximale puis on les affiche
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        
        min_date = df[col].min()
        max_date = df[col].max()
        
        print(f"  {col} : {min_date} → {max_date}")



Couverture temporelle des datasets


football_data_2526.csv
  Date : 2025-08-15 00:00:00 → 2026-04-13 00:00:00


C:\Users\LouisHarle\AppData\Local\Temp\ipykernel_23536\2466751618.py:16: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors="coerce")


In [ ]:
# On cherche ici le nombre d'observations par année

for file in folder.glob("*.csv"):
    df = pd.read_csv(file)
    
    # Les colonnes présentant des dates
    date_cols = [col for col in df.columns if "date" in col.lower()]
    
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        
        print(f"\n{file.name} - {col}")
        # On compte pour chaque année le nombre d'observations
        print(df[col].dt.year.value_counts().sort_index())



football_data_2526.csv - Date
Date
2025    802
2026    666
Name: count, dtype: int64


C:\Users\LouisHarle\AppData\Local\Temp\ipykernel_23536\3824261365.py:9: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors="coerce")


## 4.5 Biais de sélection

De potentiels biais peuvent apparaître dans les données, comme des championnats surreprésentés. Nous pouvons ainsi vérifier si ce biais est présent dans nos données.

Ce fichier .csv ne présentant pas de joueurs, il ne peut y avoir des biais que parmi les championnats représentés.

In [ ]:
# Le fichier gfootball_data_2526.csv est alors celui qui nous intéresse
data = pd.read_csv(folder / "football_data_2526.csv")

# On compte le nombre de matches par championnat
league_counts = data["league"].value_counts()

print("\nTop compétitions :\n")
print(league_counts.head(20))



Top compétitions :

league
I1     320
E0     319
SP1    310
D1     261
F1     258
Name: count, dtype: int64


In [ ]:
# Pour y voir plus clair (les noms des championnats), nous renommons les noms des ligues

mapping = {
    "E0": "Premier League",
    "SP1": "Liga",
    "I1": "Serie A",
    "F1": "Ligue 1",
    "D1": "Bundesliga"
}

# On remplacer les noms dans la colonne 'league'
data["league"] = data["league"].replace(mapping)

# On recalcule les statistiques avec les nouveaux noms
league_counts = data["league"].value_counts()

print("\nTop compétitions :\n")
print(league_counts.head(20))


Top compétitions :

league
Serie A           320
Premier League    319
Liga              310
Bundesliga        261
Ligue 1           258
Name: count, dtype: int64


## 4.6 La date de dernière mise à jour

Pour constater si nos données sont mises à jour régulièrement, nous pouvons regarder la date de dernière mise à jour.

In [ ]:
# On choisit la saison et les ligues qui nous intéressent
season = "2526"
leagues = {
    "E0": "Premier League",
    "SP1": "Liga",
    "I1": "Serie A",
    "F1": "Ligue 1",
    "D1": "Bundesliga"
}

print(f"Mises à jour football-data.co.uk\n")

updates = []

for code, name in leagues.items():
    url = f"https://www.football-data.co.uk/mmz4281/{season}/{code}.csv"
    
    # Requête directe des en-têtes
    response = requests.head(url, timeout=10)
    last_mod = response.headers.get('Last-Modified')
    
    updates.append({
        "Compétition": name,
        "Dernière mise à jour": last_mod
    })

# Affichage
df_updates = pd.DataFrame(updates)
print(df_updates.to_string(index=False))

Mises à jour football-data.co.uk

   Compétition          Dernière mise à jour
Premier League Tue, 21 Apr 2026 10:24:27 GMT
          Liga Tue, 14 Apr 2026 10:25:04 GMT
       Serie A Tue, 21 Apr 2026 10:24:26 GMT
       Ligue 1 Mon, 20 Apr 2026 14:59:14 GMT
    Bundesliga Mon, 20 Apr 2026 14:59:13 GMT
